#### Engine Design

- **Portfolio:** The Ledger's responsibility is State Integrity.
- - It must answer one question at any millisecond: "If we shut down the bot right now, how much money did we actually make?"
  - Requirements: Realized PnL (from trade), Unrealized PnL (holdings), Inventory, history_log (from trades)
- **ExecutionEngine:** For this project, this is where we get our matched orders from market. And we execute those.
- - The Engine's responsibility is Matching Logic. It acts as the "Exchange."
  - Requirements: It looks at the incoming Market Orders and decides if they "hit" your limit orders.

### Overall Flow of engine
1. We process market/limit orders on exchange OrderBook - from our CSV data (replay can be streamed).
2. While processing the orders, we keep our engine in place which is responsible for below actions:
   - Observe market signals (OBI, micro price, running_volatilty, keyle's lambda, etc)
   - Adjust our quotes based on factors like inventory, time_horizon, gamma, kappa
   - Cancel/Replace limit orders on exchange (OrderBook) quickly to avoid adverse selection with new prices based on information from signals.
   - Ledger updates if get filled (Portfolio)
   - Show after each trade, realized PnL and unrealized PnL of our portfolio.

##### Assumptions
In real exchanges, orders are filled based on Price-Time Priority:
   - Price: Best price goes first.
   - Time: If prices are equal, the person who got there first goes first - FIFO
   - Here in this project: **Assume we are always on top of book**

In [2]:
import sys
import os

target_dir = os.path.abspath('../src/')
sys.path.insert(0, target_dir) 

In [3]:
#!pip install pandas
import pandas as pd

market_orders = pd.read_csv('./../src/data/market_data.csv')
market_orders

,timestamp,type,side,price,size
0,09:30:00.001,L,B,100.00,100
1,09:30:00.002,L,S,100.10,100
2,09:30:00.005,L,B,100.01,50
3,09:30:00.010,M,S,100.01,20
4,09:30:00.015,M,S,100.01,40
5,09:30:00.020,L,S,100.08,100
6,09:30:00.025,M,B,100.08,50
7,09:30:00.030,L,B,100.02,100
8,09:30:00.035,M,S,100.02,110
9,09:30:00.040,L,S,100.05,50


In [4]:
for e in market_orders.iterrows():
    print(e)
    break

(0, timestamp    09:30:00.001
type                    L
side                    B
price               100.0
size                  100
Name: 0, dtype: object)


In [5]:
from portfolio import Portfolio
from execution_engine import ExecutionEngine

portfolio = Portfolio()
engine = ExecutionEngine(portfolio)

# plain - without adjusting bi
for _, event in market_orders.iterrows():
    if event['type'] == 'M':# == 'MarketOrder':
        report = engine.process_market_event(
            event['side'], event['price'], event['size'], 
            99.98, 100.02
        )
        if report: print(report) 
        stats = portfolio.get_metrics(100.00)
        print(stats)

{'inventory': 0, 'total_pnl': 0.0}
{'inventory': 0, 'total_pnl': 0.0}
FILLED SELL: 50 @ 100.02
{'inventory': -50, 'total_pnl': 1.0}
{'inventory': -50, 'total_pnl': 1.0}
FILLED SELL: 10 @ 100.02
{'inventory': -60, 'total_pnl': 1.2}
FILLED BUY: 50 @ 99.98
{'inventory': -10, 'total_pnl': 2.2}
FILLED BUY: 50 @ 99.98
{'inventory': 40, 'total_pnl': 3.2}
{'inventory': 40, 'total_pnl': 3.2}
FILLED BUY: 10 @ 99.98
{'inventory': 50, 'total_pnl': 3.4}


In [6]:
from pricing_engine import ASPricingEngine
from orderbook import OrderBook

portfolio = Portfolio()
engine = ExecutionEngine(portfolio)
as_pricing = ASPricingEngine(gamma=0.1, kappa=1.5)
ob = OrderBook()

In [19]:
import pandas as pd
from orderbook import OrderBook
from portfolio import Portfolio
from pricing_engine import ASPricingEngine
from execution_engine import ExecutionEngine
from math_utils import Signals

def run_backtest(csv_path):
    # 1. Initialize Components
    ob = OrderBook() # The "Public" State - exchange what everyone sees
    portfolio = Portfolio(initial_cash=100000) # The "Private" State of our market making engine
    pricing = ASPricingEngine(gamma=0.0001, kappa=20.0) # The "Decision" Logic - kind of Brain of our engine!
    simulator = ExecutionEngine(portfolio) # The "Handshake" (Matchmaker)
    
    # Load Data
    df = pd.read_csv(csv_path)
    
    print(f"Starting Backtest on {len(df)} events...")
    prev_price, prev_vol = None, None

    # 2. The Master Event Loop
    for _, row in df.iterrows():
        # A. Update Market Senses
        # We pass the row as a tuple: (type, side, price, size)
        rem, fill = ob.process_event((row['type'], row['side'], row['price'], row['size']))
        print(ob.bids.peekitem(0) if ob.bids else None, ob.asks.peekitem(0) if ob.asks else None, rem, fill)
        
        # B. Get "Market Truth"
        mid = ob.get_micro_price()
        if mid is None: continue # Wait for liquidity
        mid = round(mid,2)

        if prev_price!=None:
            kl = abs(row['price']-prev_price)/prev_vol# keyle's lambda
            print("keyle's lambda", kl)
        else:
            kl = 0.01
        prev_price = row['price']
        prev_vol = row['size']
        
        # C. Brain Decision
        # Pricing engine looks at Mid and Portfolio Inventory
        my_bid, my_ask = pricing.get_quotes(
            mid, 
            portfolio.inventory,
            kl, # Keyle's lambda
            0.01, # Todo: volatility - make it dyanamic
            1 # Time horizon
        )
        # # Temporary check of tight spread
        # my_bid = round(mid-0.01,2)
        # my_ask = round(mid+0.01,2)
        print(my_bid, my_ask, portfolio.inventory, mid)
        
        # D. The Simulation (The Collision) - Say we get filled!
        if row['type'] == 'M':
            fill = simulator.process_market_event(
                market_side=row['side'],
                market_price=row['price'],
                market_size=row['size'],
                my_bid=my_bid,
                my_ask=my_ask
            )
            print(f"[{row['timestamp']}] TRADE: {fill}")

    # 3. Final Performance Report
    final_stats = portfolio.get_metrics(mid)
    print("\n" + "="*30)
    print(f"Backtest Complete")
    print(f"Final PnL: ${final_stats['total_pnl']}")
    print(f"Final Inv: {final_stats['inventory']}")
    print("="*30)

In [20]:
run_backtest('../src/data/market_data.csv')

Starting Backtest on 20 events...
(-100.0, 100) None 100 []
(-100.0, 100) (100.1, 100) 100 []
effect. kappa 19.801980198019802
100.0 100.1 0 100.05
(-100.01, 50) (100.1, 100) 50 []
keyle's lambda 0.000899999999999892
effect. kappa 19.98201618543311
99.99 100.09 0 100.04
(-100.01, 30) (100.1, 100) 0 [{'side': 'S', 'price': 100.01, 'size': 0}]
keyle's lambda 0.0
effect. kappa 20.0
99.98 100.08 0 100.03
[09:30:00.010] TRADE: None
(-100.0, 90) (100.1, 100) 0 [{'side': 'S', 'price': 100.01, 'size': 30}, {'side': 'S', 'price': 100.0, 'size': 0}]
keyle's lambda 0.0
effect. kappa 20.0
100.0 100.1 0 100.05
[09:30:00.015] TRADE: None
(-100.0, 90) (100.08, 100) 100 []
keyle's lambda 0.0017499999999998294
effect. kappa 19.96506114299975
99.99 100.09 0 100.04
(-100.0, 90) (100.08, 50) 0 [{'side': 'B', 'price': 100.08, 'size': 0}]
keyle's lambda 0.0
effect. kappa 20.0
100.0 100.1 0 100.05
[09:30:00.025] TRADE: None
(-100.02, 100) (100.08, 50) 100 []
keyle's lambda 0.0012000000000000454
effect. kappa